In [14]:
import numpy as np
import pandas as pd

from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))
from src.preprocessing import build_pipeline
from src.feature_engineering import add_engineered_features
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import (
    mean_squared_log_error,
    mean_absolute_error,
    mean_squared_error,
    make_scorer
)

In [15]:
from pathlib import Path
from sklearn.model_selection import train_test_split

# Locate the data directory
DATA_DIR = Path("../data")

if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = Path("data")

# Load original Kaggle datasets
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Separate predictors and target
X = train_df.drop(columns=["Id", "SalePrice"])
y = train_df["SalePrice"]

# Prepare Kaggle test predictors
X_test = test_df.drop(columns=["Id"])

# Split development data into training and validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Verify dataset dimensions
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_valid:", y_valid.shape)

# Verify that all datasets have the same original predictors
assert list(X_train.columns) == list(X_valid.columns)
assert list(X_train.columns) == list(X_test.columns)

print("\nAll datasets have compatible columns.")

X_train: (1168, 79)
X_valid: (292, 79)
X_test: (1459, 79)

y_train: (1168,)
y_valid: (292,)

All datasets have compatible columns.


In [16]:

feature_inputs = [
    "GrLivArea",
    "TotalBsmtSF",
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "YrSold",
    "YearBuilt",
    "YearRemodAdd",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "OverallQual"
]

missing_summary = (
    X_train[feature_inputs]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing_summary)


GrLivArea        0
TotalBsmtSF      0
FullBath         0
HalfBath         0
BsmtFullBath     0
BsmtHalfBath     0
BsmtFinSF1       0
BsmtFinSF2       0
YrSold           0
YearBuilt        0
YearRemodAdd     0
WoodDeckSF       0
OpenPorchSF      0
EnclosedPorch    0
3SsnPorch        0
ScreenPorch      0
OverallQual      0
dtype: int64

In [17]:
# Original data: baseline
X_train_base = X_train.copy()
X_valid_base = X_valid.copy()

# Feature Engineering without interactions
X_train_fe = add_engineered_features(
    X_train,
    include_interactions=False
)

X_valid_fe = add_engineered_features(
    X_valid,
    include_interactions=False
)

# Feature Engineering including interactions
X_train_fe_inter = add_engineered_features(
    X_train,
    include_interactions=True
)

X_valid_fe_inter = add_engineered_features(
    X_valid,
    include_interactions=True
)

# Verify dimensions
print("Baseline:", X_train_base.shape)
print("Feature Engineering:", X_train_fe.shape)
print("Feature Engineering + Interaction:", X_train_fe_inter.shape)


new_features = [
    "TotalLivingArea",
    "TotalBathrooms",
    "TotalFinishedBsmtSF",
    "HouseAgeAtSale",
    "YearsSinceRemodel",
    "TotalOutdoorArea",
    "QualityAreaInteraction"
]

display(X_train_fe_inter[new_features].describe().T)


Baseline: (1168, 79)
Feature Engineering: (1168, 85)
Feature Engineering + Interaction: (1168, 86)


,count,mean,std,min,25%,50%,75%,max
TotalLivingArea,1168.0,2589.172945,823.125867,334.0,2041.0,2500.0,3028.25,11752.0
TotalBathrooms,1168.0,2.222175,0.779502,1.0,2.0,2.0,2.50,6.0
TotalFinishedBsmtSF,1168.0,491.176370,478.996643,0.0,0.0,482.5,803.25,5644.0
HouseAgeAtSale,1168.0,36.852740,30.703110,0.0,7.0,35.0,55.00,136.0
YearsSinceRemodel,1168.0,22.922089,20.713616,0.0,4.0,14.0,42.00,60.0
TotalOutdoorArea,1168.0,186.584760,161.131777,0.0,45.0,168.0,275.25,1027.0
QualityAreaInteraction,1168.0,16585.607877,8680.050912,334.0,10737.5,15019.5,20217.75,117520.0


In [18]:

def calculate_rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )


rmsle_scorer = make_scorer(
    calculate_rmsle,
    greater_is_better=False
)


In [19]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

def evaluate_svr(name, X_tr, X_val):

    svr = SVR(
        kernel="rbf",
        C=5.342937261279771,
        epsilon=0.004678719265016202,
        gamma=0.001
    )

    pipeline = TransformedTargetRegressor(
        regressor=build_pipeline(
            X_tr,
            svr,
            scale_numeric=True
        ),
        func=np.log1p,
        inverse_func=np.expm1
    )

    # Cross-validation using the same CV configuration as Phase 4.
    scores = cross_val_score(
        pipeline,
        X_tr,
        y_train,
        scoring=rmsle_scorer,
        cv=cv,
        n_jobs=-1
    )

    # Holdout validation
    pipeline.fit(X_tr, y_train)
    predictions = pipeline.predict(X_val)

    results = {
        "Dataset": name,
        "CV RMSLE": -scores.mean(),
        "CV Std": scores.std(),
        "Validation RMSLE": calculate_rmsle(
            y_valid,
            predictions
        ),
        "Validation MAE": mean_absolute_error(
            y_valid,
            predictions
        ),
        "Validation RMSE": np.sqrt(
            mean_squared_error(y_valid, predictions)
        )
    }

    return results


In [20]:

experiments = [
    ("Baseline", X_train_base, X_valid_base),

    ("Feature Engineering", X_train_fe, X_valid_fe),

    (
        "Feature Engineering + Interaction",
        X_train_fe_inter,
        X_valid_fe_inter
    )
]

results = []

for name, X_tr, X_val in experiments:

    print(f"Evaluating: {name}")

    result = evaluate_svr(name, X_tr, X_val)

    results.append(result)

results_df = pd.DataFrame(results)

display(
    results_df.sort_values("CV RMSLE")
)


Evaluating: Baseline
Evaluating: Feature Engineering
Evaluating: Feature Engineering + Interaction


,Dataset,CV RMSLE,CV Std,Validation RMSLE,Validation MAE,Validation RMSE
2,Feature Engineering + Interaction,0.116556,0.018179,0.130848,14396.243913,24418.857372
1,Feature Engineering,0.116832,0.018418,0.130714,14345.936446,24214.451900
0,Baseline,0.117979,0.018991,0.131341,14419.837259,24103.547942


In [21]:

# Six engineered features, excluding the interaction
engineered_features = [
    "TotalLivingArea",
    "TotalBathrooms",
    "TotalFinishedBsmtSF",
    "HouseAgeAtSale",
    "YearsSinceRemodel",
    "TotalOutdoorArea"
]


def run_ablation_experiment(
    name,
    include_interactions=False,
    features_to_remove=None
):
    """
    Evaluate SVR after removing selected engineered features.
    """

    if features_to_remove is None:
        features_to_remove = []

    # Generate engineered features
    X_tr = add_engineered_features(
        X_train,
        include_interactions=include_interactions
    )

    X_val = add_engineered_features(
        X_valid,
        include_interactions=include_interactions
    )

    # Remove selected features
    if features_to_remove:
        X_tr = X_tr.drop(columns=features_to_remove)
        X_val = X_val.drop(columns=features_to_remove)

    # Evaluate using the existing SVR configuration
    result = evaluate_svr(name, X_tr, X_val)

    result["Removed features"] = (
        ", ".join(features_to_remove)
        if features_to_remove
        else "None"
    )

    return result


# 1. Baseline
ablation_results = []

baseline_result = evaluate_svr(
    "Baseline",
    X_train.copy(),
    X_valid.copy()
)

baseline_result["Removed features"] = "N/A"
ablation_results.append(baseline_result)


# 2. All six engineered features
ablation_results.append(
    run_ablation_experiment(
        "Full Feature Engineering"
    )
)


# 3. Six features + interaction
ablation_results.append(
    run_ablation_experiment(
        "FE + Interaction",
        include_interactions=True
    )
)


# 4-9. Remove one engineered feature at a time
for feature in engineered_features:

    result = run_ablation_experiment(
        name=f"Without {feature}",
        features_to_remove=[feature]
    )

    ablation_results.append(result)


# Display results
ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df.sort_values("CV RMSLE").reset_index(drop=True)
)


,Dataset,CV RMSLE,CV Std,Validation RMSLE,Validation MAE,Validation RMSE,Removed features
0,FE + Interaction,0.116556,0.018179,0.130848,14396.243913,24418.857372,None
1,Full Feature Engineering,0.116832,0.018418,0.130714,14345.936446,24214.451900,None
2,Without YearsSinceRemodel,0.116911,0.018396,0.130946,14370.643382,24234.486943,YearsSinceRemodel
3,Without HouseAgeAtSale,0.116913,0.018427,0.130868,14369.359313,24260.719711,HouseAgeAtSale
4,Without TotalBathrooms,0.116926,0.018327,0.130950,14368.100834,24271.758795,TotalBathrooms
5,Without TotalFinishedBsmtSF,0.116934,0.018388,0.130704,14370.942528,24194.451655,TotalFinishedBsmtSF
6,Without TotalOutdoorArea,0.116998,0.018602,0.130756,14371.761357,24282.544870,TotalOutdoorArea
7,Without TotalLivingArea,0.117312,0.018876,0.130662,14300.027099,23949.536613,TotalLivingArea
8,Baseline,0.117979,0.018991,0.131341,14419.837259,24103.547942,N/A



# Phase 6 — Feature Engineering: Conclusions

The Feature Engineering phase introduced six domain-based features
and one interaction feature based on the EDA findings.

The experiments showed that the engineered features improved
the SVR model compared with the original baseline.

The feature ablation experiment indicated that removing individual
engineered features resulted in small performance decreases.
Therefore, all engineered features were retained.

The interaction between OverallQual and total area achieved
the best cross-validation RMSLE, although its holdout validation
performance was slightly worse than the version without interaction.

## Final results

| Metric | Baseline | Feature Engineering + Interaction |
|---|---:|---:|
| CV RMSLE | 0.117979 | 0.116556 |
| Validation RMSLE | 0.131341 | 0.130848 |

The optimized SVR with all engineered features will be used
as a candidate model in the next phase.

Hyperparameter optimization and model ensembling will be
explored separately in Phase 7.
